# Machine Learning - Previsão da Taxa de Crimes

Este notebook treina modelos de regressão para prever a taxa de crimes por 100 mil habitantes nas capitais brasileiras.

Fonte oficial:

`datamart.vw_base_modelagem_ml`

Estratégia:

- **Modelo A - Baseline sem educação**: usa histórico criminal, população e IDHM.
- **Modelo B - Principal com educação**: adiciona IDEB, fluxo, aprendizado e notas.
- **Modelo B Ridge**: versão regularizada do modelo principal.

A comparação entre o Modelo A e o Modelo B ajuda a avaliar se os indicadores educacionais melhoram a previsão.

## 1. Imports

In [ ]:
import os

import numpy as np
import pandas as pd

from sqlalchemy import create_engine

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 2. Conexão com o PostgreSQL

Dentro do Docker Compose, o host do PostgreSQL é `postgres-service`.

Se o notebook for executado fora do container, altere `POSTGRES_HOST` para `localhost`.

In [ ]:
DB_USER = os.getenv('POSTGRES_USER', 'postgres')
DB_PASSWORD = os.getenv('POSTGRES_PASSWORD', 'postgres')
DB_HOST = os.getenv('POSTGRES_HOST', 'postgres-service')
DB_PORT = os.getenv('POSTGRES_PORT', '5432')
DB_NAME = os.getenv('POSTGRES_DB', 'seguranca_publica')

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

## 3. Carregar base de modelagem

A view abaixo foi criada para Machine Learning. Ela já traz o target do ano atual e variáveis criminais defasadas (`lag1`) para evitar vazamento de informação.

In [ ]:
query = """
SELECT *
FROM datamart.vw_base_modelagem_ml
ORDER BY codigo_municipio, ano;
"""

df_raw = pd.read_sql(query, engine)
df_raw.head()

## 4. Diagnóstico inicial

In [ ]:
display(df_raw.shape)
display(df_raw.dtypes)
display(df_raw.isna().sum().sort_values(ascending=False))
display(df_raw.groupby('ano').size().rename('qtd_linhas'))

## 5. Preparação da base

Como o IDEB não é divulgado todos os anos, usamos `forward fill` por UF para propagar o último valor educacional conhecido.

Isso não usa informação futura: cada ano recebe apenas o último valor disponível até aquele momento.

In [ ]:
df = df_raw.copy()
df = df.sort_values(['sigla_uf', 'ano']).reset_index(drop=True)

colunas_numericas = [
    'ano',
    'populacao_total',
    'populacao_crescimento_pct',
    'idhm',
    'idhm_renda',
    'idhm_educacao',
    'idhm_longevidade',
    'ideb',
    'fluxo',
    'aprendizado',
    'nota_mt',
    'nota_lp',
    'taxa_crimes_100k_lag1',
    'taxa_mortes_violentas_100k_lag1',
    'risco_indice_lag1',
    'target_taxa_crimes_100k',
]

for coluna in colunas_numericas:
    df[coluna] = pd.to_numeric(df[coluna], errors='coerce')

colunas_educacao = ['ideb', 'fluxo', 'aprendizado', 'nota_mt', 'nota_lp']

for coluna in colunas_educacao:
    df[f'{coluna}_preenchido'] = df.groupby('sigla_uf')[coluna].ffill()

df[['ano', 'sigla_uf', 'ideb', 'ideb_preenchido']].head(12)

## 6. Definição de features e target

In [ ]:
target = 'target_taxa_crimes_100k'

features_baseline = [
    'ano',
    'populacao_total',
    'populacao_crescimento_pct',
    'idhm',
    'idhm_renda',
    'idhm_educacao',
    'idhm_longevidade',
    'taxa_crimes_100k_lag1',
    'taxa_mortes_violentas_100k_lag1',
    'risco_indice_lag1',
]

features_educacao = features_baseline + [
    'ideb_preenchido',
    'fluxo_preenchido',
    'aprendizado_preenchido',
    'nota_mt_preenchido',
    'nota_lp_preenchido',
]

colunas_identificacao = ['ano', 'codigo_municipio', 'nome_municipio', 'sigla_uf', 'nome_regiao']

features_baseline, features_educacao

## 7. Split temporal

Plano ideal quando a base estiver atualizada até 2025:

- Treino: 2017, 2018 e 2019
- Teste: 2023, 2024 e 2025
- Excluir: 2020, 2021 e 2022

Se esses anos ainda não existirem na base, o notebook usa uma divisão temporal alternativa para permitir validação durante o desenvolvimento.

In [ ]:
anos_treino_planejado = [2017, 2018, 2019]
anos_teste_planejado = [2023, 2024, 2025]
anos_excluir = [2020, 2021, 2022]

def escolher_split_temporal(df_modelo, features, target):
    colunas_modelo = list(dict.fromkeys(colunas_identificacao + features + [target]))
    dados = df_modelo[colunas_modelo].dropna().copy()
    anos_disponiveis = set(dados['ano'].astype(int).unique())

    tem_split_planejado = (
        set(anos_treino_planejado).issubset(anos_disponiveis)
        and len(set(anos_teste_planejado).intersection(anos_disponiveis)) > 0
    )

    if tem_split_planejado:
        treino = dados[dados['ano'].isin(anos_treino_planejado)].copy()
        teste = dados[dados['ano'].isin(anos_teste_planejado)].copy()
        estrategia = 'split_planejado_pos_pandemia'
    else:
        dados_sem_anos_excluidos = dados[~dados['ano'].isin(anos_excluir)].copy()
        anos_validos = sorted(dados_sem_anos_excluidos['ano'].astype(int).unique())

        if len(anos_validos) >= 2:
            ano_teste = anos_validos[-1]
            treino = dados_sem_anos_excluidos[dados_sem_anos_excluidos['ano'] < ano_teste].copy()
            teste = dados_sem_anos_excluidos[dados_sem_anos_excluidos['ano'] == ano_teste].copy()
            estrategia = f'fallback_temporal_teste_{ano_teste}'
        else:
            anos_validos = sorted(dados['ano'].astype(int).unique())
            ano_teste = anos_validos[-1]
            treino = dados[dados['ano'] < ano_teste].copy()
            teste = dados[dados['ano'] == ano_teste].copy()
            estrategia = f'fallback_com_pandemia_teste_{ano_teste}'

    return treino, teste, estrategia

treino_a, teste_a, estrategia_a = escolher_split_temporal(df, features_baseline, target)
treino_b, teste_b, estrategia_b = escolher_split_temporal(df, features_educacao, target)

resumo_split = pd.DataFrame([
    {
        'modelo': 'Modelo A - Baseline sem educacao',
        'estrategia': estrategia_a,
        'anos_treino': sorted(treino_a['ano'].unique()),
        'anos_teste': sorted(teste_a['ano'].unique()),
        'linhas_treino': len(treino_a),
        'linhas_teste': len(teste_a),
    },
    {
        'modelo': 'Modelo B - Principal com educacao',
        'estrategia': estrategia_b,
        'anos_treino': sorted(treino_b['ano'].unique()),
        'anos_teste': sorted(teste_b['ano'].unique()),
        'linhas_treino': len(treino_b),
        'linhas_teste': len(teste_b),
    },
])

resumo_split

## 8. Funções de treino e avaliação

In [ ]:
def treinar_e_avaliar(nome_modelo, estimador, treino, teste, features, target):
    X_train = treino[features]
    y_train = treino[target]
    X_test = teste[features]
    y_test = teste[target]

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', estimador),
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    metricas = {
        'modelo': nome_modelo,
        'mae': mean_absolute_error(y_test, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
        'r2': r2_score(y_test, y_pred) if len(y_test) > 1 else np.nan,
        'linhas_treino': len(treino),
        'linhas_teste': len(teste),
    }

    previsoes = teste[colunas_identificacao].copy()
    previsoes['modelo'] = nome_modelo
    previsoes['valor_real'] = y_test.values
    previsoes['valor_previsto'] = y_pred
    previsoes['erro'] = previsoes['valor_real'] - previsoes['valor_previsto']
    previsoes['erro_absoluto'] = previsoes['erro'].abs()

    modelo = pipeline.named_steps['model']
    coeficientes = pd.DataFrame({
        'modelo': nome_modelo,
        'feature': features,
        'coeficiente': modelo.coef_,
    }).sort_values('coeficiente', ascending=False)

    return pipeline, metricas, previsoes, coeficientes

## 9. Modelo A - Baseline sem educação

In [ ]:
modelo_a, metricas_a, previsoes_a, coeficientes_a = treinar_e_avaliar(
    nome_modelo='Modelo A - Linear sem educacao',
    estimador=LinearRegression(),
    treino=treino_a,
    teste=teste_a,
    features=features_baseline,
    target=target,
)

pd.DataFrame([metricas_a])

## 10. Modelo B - Principal com educação

In [ ]:
modelo_b, metricas_b, previsoes_b, coeficientes_b = treinar_e_avaliar(
    nome_modelo='Modelo B - Linear com educacao',
    estimador=LinearRegression(),
    treino=treino_b,
    teste=teste_b,
    features=features_educacao,
    target=target,
)

pd.DataFrame([metricas_b])

## 11. Modelo B Ridge - Versão regularizada

A Ridge Regression ajuda a estabilizar o modelo quando existem variáveis correlacionadas, como IDHM, IDEB, aprendizado e notas.

In [ ]:
modelo_b_ridge, metricas_b_ridge, previsoes_b_ridge, coeficientes_b_ridge = treinar_e_avaliar(
    nome_modelo='Modelo B - Ridge com educacao',
    estimador=Ridge(alpha=1.0),
    treino=treino_b,
    teste=teste_b,
    features=features_educacao,
    target=target,
)

pd.DataFrame([metricas_b_ridge])

## 12. Comparação dos modelos

In [ ]:
metricas_modelos = pd.DataFrame([metricas_a, metricas_b, metricas_b_ridge])
metricas_modelos.sort_values('rmse')

## 13. Previsões e maiores erros

In [ ]:
previsoes_modelos = pd.concat(
    [previsoes_a, previsoes_b, previsoes_b_ridge],
    ignore_index=True,
)

previsoes_modelos.sort_values(['modelo', 'erro_absoluto'], ascending=[True, False]).head(20)

## 14. Interpretação dos coeficientes

In [ ]:
coeficientes_modelos = pd.concat(
    [coeficientes_a, coeficientes_b, coeficientes_b_ridge],
    ignore_index=True,
)

coeficientes_modelos

## 15. Conclusões iniciais

Pontos para interpretar após executar o notebook:

- O Modelo B reduziu MAE/RMSE em relação ao Modelo A?
- O Ridge melhorou a estabilidade do modelo principal?
- Quais capitais tiveram maior erro absoluto?
- Os coeficientes fazem sentido com a hipótese do projeto?
- A inclusão de educação agregou poder preditivo ou apenas complexidade?

Observação: com a base atual, ainda temos poucos anos. Quando os dados até 2025 forem carregados no DW/Data Mart, este notebook deve ser reexecutado para usar o split planejado pós-pandemia.